# q_C 강도 × q_V 가치기저 M5: Dunnhumby seed 42

새 M5 한 개만 100 epoch 학습하는 빠른 개발 스크리닝입니다. 사용자 경제표현은 `q_C(u) × b(q_V(u))`, 아이템은 `b(구매금액 백분위)`이고, 두 표현을 ID 임베딩과 함께 binary LightGCN에서 전파하고 동시 학습합니다. 다른 `q_N` 게이트는 없고, `N`은 `q_C=percentile(n_u×v_u)` 내부에서만 반영됩니다.

M4는 기존 개인별 양성가중을 수정 없이 유지합니다. 이전 M1(`7cd818302fb1`)과 M4(`d01883236772`)는 파일을 불러오지 않고 검증된 수치로만 참고 비교하므로, 과거 결과 JSON이 없어도 실행이 멈추지 않습니다. 단, 서로 다른 실행의 참고 비교이므로 정식 요인효과나 CLV 귀속을 판정하지 않습니다. final test와 holdout은 만들지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

PINNED_SOURCE_COMMIT = '14afb9254c017541830837f42b26ff073bef18d2'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(
    ['git', '-C', str(repo), 'checkout', '-q', PINNED_SOURCE_COMMIT],
    check=True,
)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == PINNED_SOURCE_COMMIT, (actual_sha, PINNED_SOURCE_COMMIT)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_clv_scaled_value_basis_one_arm import (
    MODEL_ID,
    configure_clv_scaled_value_basis_one_arm,
    preflight_summary,
    run_clv_scaled_value_basis_one_arm,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
data_root = Path('/content/drive/MyDrive') / '논문' / 'data'
cfg = configure_clv_scaled_value_basis_one_arm(
    out_dir=str(data_root / 'results_v3_dunnhumby_m5_clv_scaled_value_basis_one_arm_development_screen_v1'),
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == [MODEL_ID]
assert summary['reused_models'] == []
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
assert summary['fixed']['external_reranking'] is False
assert summary['fixed']['m3_edge_weight'] is False
assert summary['m2']['separate_q_n_gate'] is False
assert summary['comparison']['missing_reference_file_can_block_run'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_clv_scaled_value_basis_one_arm(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

main_metrics = [
    'model_id', 'role', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10',
    'vndcg@10', 'coverage@10',
]
print('1) 새 M5 핵심 절대지표')
show(result_df[[column for column in main_metrics if column in result_df.columns]])
print('2) 이전 M1·M4 참고값과 핵심지표 비교(서로 다른 실행)')
show(result_df.attrs['reference_comparison'])
print('3) ID 점수 대비 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) 사용자 내 후보 구분·Top-10 변경 진단')
print(json.dumps(result_df.attrs['rank_intervention_diagnostics'], ensure_ascii=False, indent=2))
show(result_df.attrs['top10_overlap'])
print('5) 방향성 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('6) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))